Eigentdata初步架构构想（大部分都可以基于camel去做）：
selfinstruct+graphrag+multi-agents+function call：
1、用户上传的数据(url+firecrawl/pdf/docx)给unstructured.io转成chunks
2、chunks给qdrant向量数据库+chunks总结为知识点代替知识图谱，并以某种json格式输出
3、生成知识图谱（可选 成本高）
4、instruct agent：遍历知识图谱每个关键点+graphrag生成多条instructions（可加更高频率调用functioncall的instructions）
5、response agent：负责基于instruct agent的output中生成的每条instructions和rag生成对应的output（基于CaelumF的function call data gene可以生成）
6、convert 每条指令为sharegpt or Alpaca格式
7、score agent：对每条指令数据集进行评分


用户上传的数据(url+firecrawl/pdf/docx)给unstructured.io转成chunks

chunks给向量数据库+chunks总结为知识点代替知识图谱，并以某种json格式输出

生成知识图谱（可选 成本高）

instruct agent：遍历知识图谱每个关键点+graphrag生成多条instructions

input：请你基于以下知识点（前面chunks总结的知识点或直接生成的知识图谱），检索用户上传的数据，生成{数目}极其相关的高质量问题，并以某种json格式输出
output：多条instructions

example：
input：请你基于以下知识点：{乙肝}，检索用户上传的数据，生成{数目}极其相关的高质量问题
output：乙肝的定义、乙肝的传播途径、乙肝的预防、乙肝的治疗、乙肝的并发症、乙肝的预后






In [ ]:
#  Single Agent with Auto RAG

from camel.agents import ChatAgent
from camel.messages import BaseMessage
from camel.types import RoleType
from camel.retrievers import AutoRetriever
from camel.types import StorageType

def single_agent(query: str) ->str :
    # Set agent role
    assistant_sys_msg = """请你基于以下知识点（前面chunks总结的知识点或直接生成的知识图谱），检索用户上传的数据，生成{数目}极其相关的高质量问题，并以某种json格式输出"""

    # Add auto retriever
    auto_retriever = AutoRetriever(
            vector_storage_local_path="local_data2/",
            storage_type=StorageType.QDRANT,
            embedding_model=embedding_instance)

    retrieved_info = auto_retriever.run_vector_retriever(
        query=query,
        contents=[
            "local_data/camel_paper.pdf",  # example local path
            "https://github.com/camel-ai/camel/wiki/Contributing-Guidlines",  # example remote url
        ],
        top_k=1,
        return_detailed_info=False,
        similarity_threshold=0.5
    )

    # Pass the retrieved infomation to agent
    user_msg = str(retrieved_info)
    agent = ChatAgent(assistant_sys_msg)

    # Get response
    assistant_response = agent.step(user_msg)
    return assistant_response.msg.content

print(single_agent("这里直接输入用户的另外一些对应问题的生成要求"))

response agent：负责基于instruct agent的output中生成的每条instructions和rag生成对应的output
input：请你根据以下问题：{instruct agent的output}，检索用户上传的数据，生成{数目}高质量的回答，并以某种json格式输出
output：rag或graghrag的output

example：
input：请你根据以下问题：{乙肝的定义}，检索用户上传的数据，生成{数目}高质量的回答，并以某种json格式输出
output：
    {
        "乙肝的定义": "乙型肝炎是由乙型肝炎病毒（HBV）引起的一种传染病，主要通过血液传播。"
        
    }



In [ ]:
#  Single Agent with Auto RAG

from camel.agents import ChatAgent
from camel.messages import BaseMessage
from camel.types import RoleType
from camel.retrievers import AutoRetriever
from camel.types import StorageType

def single_agent(query: str) ->str :
    # Set agent role
    assistant_sys_msg = """请你根据以下问题：{instruct agent的output}，检索用户上传的数据，生成{数目}高质量的回答，并以某种json格式输出"""

    # Add auto retriever
    auto_retriever = AutoRetriever(
            vector_storage_local_path="local_data2/",
            storage_type=StorageType.QDRANT,
            embedding_model=embedding_instance)

    retrieved_info = auto_retriever.run_vector_retriever(
        query=query,
        contents=[
            "local_data/camel_paper.pdf",  # example local path
            "https://github.com/camel-ai/camel/wiki/Contributing-Guidlines",  # example remote url
        ],
        top_k=1,
        return_detailed_info=False,
        similarity_threshold=0.5
    )

    # Pass the retrieved infomation to agent
    user_msg = str(retrieved_info)
    agent = ChatAgent(assistant_sys_msg)

    # Get response
    assistant_response = agent.step(user_msg)
    return assistant_response.msg.content

print(single_agent("这里直接输入用户的另外一些对应回答的生成要求"))

方案一：score agent：对每条指令数据集进行评分
input：instruct agent的每条instructions+response agent的output
output：评分

example：
input：乙肝的定义
output：乙肝的定义：乙型肝炎是由乙型肝炎病毒（HBV）引起的一种传染病，主要通过血液传播。
评分：5分


advice：引入用户的评分机制生成评分agent的设定

————————————————————————————


方案二：filtrate agent：对instructions数据对进行
Deduplicator
RougeSelector
LengthSelector
GPTScoreSelector
MTLDSelector
RandomSelector
方法等筛选高质量的instructions
参考：https://github.com/zjunlp/EasyInstruct

input：instruct agent的output+response agent的output数据集
output：筛选后的instructions


把所有messages转换为sharegpt or Alpaca格式（待集成）